# Classification de cinq types de cancer — notebook corrigé

La structure et les analyses du notebook d’origine sont conservées : exploration, ACP, LDA, k-NN, arbres, forêt, AdaBoost, puis BRCA contre OTHER.

**Pour exécuter :** place `data.csv` et `labels.csv` dans le dossier de travail, ou indique leur dossier dans `DATA_DIR` (première cellule de code). Exécute ensuite les cellules dans l’ordre.

Le tirage à 50 % de l’analyse d’origine est conservé (`p = 0.5`, graine `74909311`). Le découpage train/test est désormais stratifié à 80/20. Les prétraitements des modèles sont ajustés dans les plis de validation croisée. Les graphiques exploratoires par gène et les ACP descriptives utilisent le train. Les figures et métriques doivent être recalculées ; les anciennes sorties ont été effacées.

Les grilles d’arbres et de forêts ont été corrigées et les scores de sélection sont le F1 macro en multiclasses, le F1 BRCA en binaire. Les comparaisons prévues sur le test sont finales : ne pas réajuster les paramètres en fonction de leurs résultats. Les scores CV après sélection ne constituent pas une estimation indépendante de cette sélection. Le jeu ayant déjà été exploré, ce découpage interne ne remplace pas une validation externe.

Vérification du code : exécution des deux tâches et de toutes les familles sur les gènes du fichier, avec des grilles réduites. Les recherches complètes restent à exécuter.

<a href="https://colab.research.google.com/github/LeoPich/Projects/blob/main/Machine_Learning_Projects/Classification%20of%20five%20cancer%20types%20from%20gene%20expression%20data/Programme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dossier contenant data.csv et labels.csv
from pathlib import Path

# Si les deux CSV sont dans le dossier de travail, ne change rien.
# Sinon, remplace Path.cwd() par Path("/chemin/vers/ton/dossier").
DATA_DIR = Path.cwd()
N_JOBS = 2  # Mettre 1 si la mémoire disponible est limitée.

for nom in ("data.csv", "labels.csv"):
    if not (DATA_DIR / nom).is_file():
        raise FileNotFoundError(
            f"{nom} introuvable dans {DATA_DIR}. Modifie DATA_DIR ci-dessus."
        )
print("Dossier des données :", DATA_DIR)


In [ ]:
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statistics import pstdev, variance, pvariance,stdev, mean
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import  linkage, fcluster
from sklearn.cluster import  KMeans
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import neighbors
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score,f1_score,make_scorer
import warnings
warnings.resetwarnings()

nb_binome = 74909311 # nombre à remplacer par votre numéro de binome
np.random.seed(nb_binome)

# Lecture unique ; on conserve tous les gènes et leurs noms d’origine.
df_donnees = pd.read_csv(DATA_DIR / "data.csv", index_col=0)
df_labels = pd.read_csv(DATA_DIR / "labels.csv", index_col=0)
assert df_donnees.index.is_unique and df_labels.index.is_unique
assert set(df_donnees.index) == set(df_labels.index)
df_labels = df_labels.loc[df_donnees.index]
assert df_labels["Class"].notna().all()

feature_names = df_donnees.columns.to_numpy()
X = df_donnees.to_numpy(dtype=float)
label = df_labels["Class"].to_numpy()
assert np.isfinite(X).all(), "Présence de valeurs manquantes ou infinies."

# Même règle que dans l’analyse d’origine : garder chaque individu
# avec une probabilité de 0.5, sans exclure la dernière ligne.
n = len(X)
p = 0.5
Keeprows = np.where(np.random.uniform(size=n) < p)[0]
X = X[Keeprows]
label = label[Keeprows]
data = X
print(len(Keeprows))
print(data.shape)

# Un seul découpage, avant les graphiques et le choix des modèles.
X_train, X_test, y_train, y_test = train_test_split(
    X, label, test_size=0.2, stratify=label, random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Train :", X_train.shape, "| Test :", X_test.shape)




print(data[0])
print(label)

sns.countplot(x=label, color="steelblue")



### Comprehension des données (graphiques sur le train) --------------------------------------------------


### couple qui marche (9176,3540) , (2,3), (3,9176)
plt.figure()
d = {'X0': X_train[:,3], 'X1': X_train[:,9176], 'Label':y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="X0",y="X1",hue="Label",data = df)
plt.show()

plt.figure()
d = {'X0': X_train[:,9176], 'X1': X_train[:,3540], 'Label':y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="X0",y="X1",hue="Label",data = df)
plt.show()

plt.figure()
d = {'X0': X_train[:,1], 'X1': X_train[:,2], 'Label':y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="X0",y="X1",hue="Label",data = df)
plt.show()
### taille des données

#print("Nombre de lignes, nombre de colonnes : ",data.shape)
n_ech=data.shape[0]
n_gen=data.shape[1]
print("Il y a "+str(n_ech)+" échantillons et "+str(n_gen)+" gênes étudiés.")

### variance des données
VarX = X_train.var(axis=0, ddof=1)
MeanX = X_train.mean(axis=0)


#Répartition PRAD - LUAD - BRCA - KIRC - COAD
def repartition():
    P,L,B,K,C=0,0,0,0,0
    for u in label:
        if u=='PRAD':
            P+=1
        elif u=='LUAD':
            L+=1
        elif u=='BRCA':
            B+=1
        elif u=='KIRC':
            K+=1
        else:
            C+=1
    print(P,L,B,K,C,n_ech)


repartition()
### Repartition des classes dans l'echantillon
plt.figure()
sns.countplot(x=label, color="steelblue")
plt.show()
### ACP et representation dans le plan des 2 premiers composantes principales ----
pca = PCA(n_components=2, svd_solver="randomized", random_state=nb_binome)
pca.fit(X_train)

PC = pca.transform(X_train)
print(PC.shape)
print(y_train.shape)

d = {'PC1': PC[:,0], 'PC2': PC[:,1], 'Label':y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="PC1",y="PC2",hue="Label",data = df)
#ACP
n_comp=20

scaler = StandardScaler(with_mean=True,with_std=False)
scaler.fit(X_train)
Xs = scaler.transform(X_train)
pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=nb_binome)
pca.fit(Xs)
inerties = pca.explained_variance_ratio_

plt.figure()
plt.bar(np.arange(1, n_comp + 1),100 * inerties,color='lightseagreen')
plt.xlabel("Composantes")
plt.ylabel("Variance expliquée (%)")
plt.title('Inerties en %'+' sans '+'standardisation')
plt.show()







In [ ]:
print("Part de variance expliquée par les 20 composantes :", sum(inerties))


In [ ]:
np.random.seed(nb_binome)
### LDA Suffisante pour classifier???---------------------------------------------
## On effectue la LDA
# Les scores ci-dessous sont calculés sur le découpage déjà créé.

### Sur echantillon train/test


# Réutilisation du train et du test définis lors du chargement.

print(' \n #### LDA SUR LES 5 CLASSES (train/test) #### \n')
lda = LinearDiscriminantAnalysis()
lda.fit(X_train,y_train)#
yhat = lda.predict(X_test)#
errl=sum(y_test!=yhat)/len(y_test) #
print("Taux d'erreur: ",round(errl,3))
plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(y_test,yhat) #
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()

score = cross_val_score(lda, X_train, y_train, cv=cv, scoring="accuracy", error_score="raise")
print("Accuracy moyenne en CV sur le train :", np.mean(score))


### KPPV ----------------------------------------------------------------------
print("### KPPV ####")

# La standardisation et le filtrage sont réappris dans chaque pli de CV.
knn_pipeline = Pipeline([
    ("variance", VarianceThreshold(threshold=1e-16)),
    ("scaler", StandardScaler()),
    ("knn", neighbors.KNeighborsClassifier()),
])

# Même première référence que dans le notebook d’origine : k = 10.
knn_pipeline.set_params(knn__n_neighbors=10)
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)
print("Matrice de confusion pour k = 10 :")
print(confusion_matrix(y_test, y_pred))
print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("F1 pondéré :", round(f1_score(y_test, y_pred, average="weighted", zero_division=0), 3))

from sklearn.metrics import roc_curve, RocCurveDisplay
y_score = knn_pipeline.predict_proba(X_test)

for i, classe in enumerate(knn_pipeline.classes_):
    fpr, tpr, _ = roc_curve(y_test == classe, y_score[:, i])
    RocCurveDisplay(fpr=fpr, tpr=tpr).plot()
    plt.title("Courbe ROC " + str(classe) + " — k = 10")
    plt.show()

# Sélection de k uniquement sur le train.
search_knn = GridSearchCV(
    knn_pipeline,
    {"knn__n_neighbors": [1, 3, 5, 10, 20]},
    scoring="f1_macro",
    cv=cv,
    n_jobs=N_JOBS,
    pre_dispatch=N_JOBS,
    error_score="raise",
)
search_knn.fit(X_train, y_train)
print("Valeurs de k :", search_knn.cv_results_["param_knn__n_neighbors"])
print("Scores CV moyens :", search_knn.cv_results_["mean_test_score"])
print("Écarts-types CV :", search_knn.cv_results_["std_test_score"])
print("Meilleur k :", search_knn.best_params_)

knn = search_knn.best_estimator_
y_pred = knn.predict(X_test)
print("Matrice de confusion pour le k retenu :")
print(confusion_matrix(y_test, y_pred))
print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("F1 pondéré :", round(f1_score(y_test, y_pred, average="weighted", zero_division=0), 3))
print("Précision pondéré :", round(precision_score(y_test, y_pred, average="weighted", zero_division=0), 3))
print("Rappel pondéré :", round(recall_score(y_test, y_pred, average="weighted", zero_division=0), 3))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="YlOrBr", colorbar=False
)
plt.show()


In [ ]:
np.random.seed(nb_binome)
### Arbre -------------------------------------------------------------------------

print('### arbre sur echantillon train/test ###')
clf = DecisionTreeClassifier(max_depth=4, random_state=nb_binome)
clf = clf.fit(X_train, y_train)

plt.figure(figsize=(30,30))
plot_tree(clf, filled=True, feature_names=feature_names, class_names=list(clf.classes_))
plt.show()

y_pred_clf = clf.predict(X_test)
print("Confusion matrix  : ")
print(confusion_matrix(y_test,y_pred_clf))
print(' ')
print("Confusion matrix (proportions) : ")
cm = confusion_matrix(y_test,y_pred_clf,normalize='true')
print(np.round(cm,2))
print(' ')

print("Accuracy : ",np.round(accuracy_score(y_test,y_pred_clf),3))
print("F1-score pondéré : ",np.round(f1_score(y_test,y_pred_clf,average='weighted'),3)) #pos_label="'BRCA'"
print("Précision pondérée : ",np.round(precision_score(y_test,y_pred_clf,average='weighted'),3)) #pos_label="'BRCA'"
print("Rappel pondéré : ",np.round(recall_score(y_test,y_pred_clf,average='weighted'),3)) #pos_label="'BRCA'"

plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(y_test,y_pred_clf) #
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()

print('### arbre sur les données d’entraînement ###')
clf = DecisionTreeClassifier(max_depth=4, random_state=nb_binome)
clf = clf.fit(X_train, y_train)

plt.figure(figsize=(30,30))
plot_tree(clf, filled=True, feature_names=feature_names, class_names=list(clf.classes_))
plt.show()


## essai avec gridsearch pour avoir les meilleurs hyperparametres

# Generation of the B training/validation sets

import warnings
# Les avertissements restent visibles.
B = 10
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "max_depth": [4, 8, None],
    "min_samples_leaf": [1, 4, 8],
    "max_features": [None, "sqrt"],
    "splitter": ["best"],
}
clf = DecisionTreeClassifier(random_state=nb_binome)
search = GridSearchCV(clf,
                      param_grid,
                      scoring="f1_macro",
                      cv=cv,
                      n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score="raise")
search.fit(X_train,y_train)
print('Meilleur modèle')
print(search.best_estimator_)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc_ovr",
}

tree = search.best_estimator_

plt.figure(figsize=(30,30))
plot_tree(tree, filled=True, feature_names=feature_names, class_names=list(tree.classes_))
plt.show()
plt.rcdefaults()

y_pred_tree = tree.predict(X_test)
print("Confusion matrix  : ")
print(confusion_matrix(y_test,y_pred_tree))
print(' ')
print("Confusion matrix (proportions) : ")
cm = confusion_matrix(y_test,y_pred_tree,normalize='true')
print(np.round(cm,2))
print(' ')

plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(y_test,y_pred_tree)
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()

print("Accuracy : ",np.round(accuracy_score(y_test,y_pred_tree),3))
print("F1-score pondéré : ",np.round(f1_score(y_test,y_pred_tree,average='weighted'),3)) #,pos_label="'BRCA'"
print("Précision pondérée : ",np.round(precision_score(y_test,y_pred_tree,average='weighted'),3)) #,pos_label="'BRCA'"
print("Rappel pondéré : ",np.round(recall_score(y_test,y_pred_tree,average='weighted'),3)) #,pos_label="'BRCA'"



## graphiques (genes 6875, 12983,15896,18746) on voit bien les séparations

#d = {'X0': X[:,3], 'X1': X[:,6875], 'Label':label}
#df = pd.DataFrame(data=d)

#sns.scatterplot(x="X0",y="X1",hue="Label",data = df)

#d = {'X0': X[:,3], 'X1': X[:,12983], 'Label':label}
#df = pd.DataFrame(data=d)

#sns.scatterplot(x="X0",y="X1",hue="Label",data = df)

#d = {'X0': X[:,3], 'X1': X[:,15896], 'Label':label}
#df = pd.DataFrame(data=d)

#sns.scatterplot(x="X0",y="X1",hue="Label",data = df)

d = {'X0': X_train[:,3], 'X1': X_train[:,18746], 'Label':y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="X0",y="X1",hue="Label",data = df)



In [ ]:
np.random.seed(nb_binome)
### Foret et bagging ------------------------------------------------------------
print("### Forêt ### ")

param_grid = {
    "max_depth": [5, None],
    "min_samples_leaf": [1, 4],
    "max_features": ["sqrt", 0.05],
}

rf = RandomForestClassifier(random_state=nb_binome)
search = GridSearchCV(rf,
                      param_grid,
                      scoring="f1_macro",
                      cv=cv,
                      n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score="raise")
search.fit(X_train,y_train)
print('Meilleur modèle')
print(search.best_estimator_)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc_ovr",
}

rf_best = search.best_estimator_
cv_scores = cross_validate(rf_best, X_train, y_train, cv=cv, scoring=scoring, error_score="raise")

print("Scores CV sur le train après sélection des paramètres")
print("Accuracy  : ", np.round(np.mean(cv_scores["test_accuracy"]),2), "(",np.round(np.std(cv_scores["test_accuracy"]),2),")")
print("Precision : ", np.round(np.mean(cv_scores["test_precision"]),2), "(",np.round(np.std(cv_scores["test_precision"]),2),")")
print("Recall    : ", np.round(np.mean(cv_scores["test_recall"]),2), "(",np.round(np.std(cv_scores["test_recall"]),2),")")
print("F1-score  : ", np.round(np.mean(cv_scores["test_f1"]),2), "(",np.round(np.std(cv_scores["test_f1"]),2),")")
print("AUC       : ", np.round(np.mean(cv_scores["test_roc_auc"]),2), "(",np.round(np.std(cv_scores["test_roc_auc"]),2),")")

rf_best = search.best_estimator_
importances = rf_best.feature_importances_
indices = np.argsort(importances)[-20:]

#Plot the feature importances of the forest
plt.figure(figsize=(9,8))
plt.title("20 gènes les plus importants dans cette forêt")
plt.barh(feature_names[indices], importances[indices], color="r", align="center")
plt.show()
plt.rcdefaults()
# # If you want to define your own labels,
# # change indices to a list of labels on the following line.
#plt.yticks(range(X.shape[1]), data.columns[indices])
#plt.ylim([-1, X.shape[1]])
#plt.savefig("RF_importances.png",format="png")
plt.show()

from sklearn.metrics import ConfusionMatrixDisplay
y_pred_rf = rf_best.predict(X_test)

print("Matrice de confusion de la forêt aléatoire :")


plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(y_test,y_pred_rf)
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()


print("Accuracy : ",np.round(accuracy_score(y_test,y_pred_rf),3))
print("F1-score pondéré : ",np.round(f1_score(y_test,y_pred_rf,average='weighted'),3))
print("Précision pondérée : ",np.round(precision_score(y_test,y_pred_rf,average='weighted'),3))
print("Rappel pondéré : ",np.round(recall_score(y_test,y_pred_rf,average='weighted'),3))



In [ ]:
### Adaboost -------------------------------------------------------------------
print("### AdaBoost ###")

# L’ACP est ajustée sur chaque pli d’entraînement, jamais sur le test.
boost_pipeline = Pipeline([
    ("variance", VarianceThreshold(threshold=1e-16)),
    ("pca", PCA(n_components=10, svd_solver="randomized", random_state=nb_binome)),
    ("boost", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, random_state=nb_binome),
        random_state=nb_binome,
    )),
])
param_grid = {
    "boost__n_estimators": [100, 200, 300],
    "boost__learning_rate": [0.01, 0.1, 1.0],
}
search = GridSearchCV(
    boost_pipeline, param_grid,
    scoring="f1_macro", cv=cv,
    n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score="raise",
)
search.fit(X_train, y_train)
print("Meilleurs paramètres :", search.best_params_)
boost_best = search.best_estimator_
y_pred_boost = boost_best.predict(X_test)

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_boost))
cm_boost = confusion_matrix(y_test, y_pred_boost, normalize="true")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_boost, cmap="YlOrBr", colorbar=False
)
plt.title("AdaBoost")
plt.show()
print("Accuracy :", round(accuracy_score(y_test, y_pred_boost), 3))
print("F1 pondéré :", round(f1_score(y_test, y_pred_boost, average="weighted", zero_division=0), 3))
print("Précision pondéré :", round(precision_score(y_test, y_pred_boost, average="weighted", zero_division=0), 3))
print("Rappel pondéré :", round(recall_score(y_test, y_pred_boost, average="weighted", zero_division=0), 3))


#### Refaire la meme chose mais avec que deux classes par exemple prendre BRCA
# Les étiquettes restent BRCA et OTHER.

### données ---------------------------------------------------------------------
Y=[]
for i in label:
    if i =='BRCA': Y.append('BRCA')
    else: Y.append('OTHER')
print(Y)
# Même découpage que pour les cinq classes.
Y_train = np.where(y_train == "BRCA", "BRCA", "OTHER")
Y_test = np.where(y_test == "BRCA", "BRCA", "OTHER")

d = {'X0': X_train[:,3], 'X1': X_train[:,6876], 'Label':Y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="X0",y="X1",hue="Label",data = df)

## repartition
plt.figure()
sns.countplot(x=Y, color="steelblue")
plt.show()

#ACP -------------------------------------------------------------------------------
pca = PCA(n_components=2, svd_solver="randomized", random_state=nb_binome)
pca.fit(X_train)

PC = pca.transform(X_train)
print(PC.shape)
print(Y_train.shape)

d = {'PC1': PC[:,0], 'PC2': PC[:,1], 'Label':Y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="PC1",y="PC2",hue="Label",data = df)

### LDA ---------------------------------------------------------------------------

# LDA entraînée uniquement sur le train ci-dessous.

### Sur echantillon train/test


# Réutilisation des mêmes individus dans le train et le test.

print("Taille de l’échantillon d’entraînement :", len(Y_train))
print("Taille de l’échantillon de test :", len(Y_test))
print(' \n #### LDA SUR LES 2 CLASSES (train/test) #### \n')
lda = LinearDiscriminantAnalysis()
lda.fit(X_train,Y_train)#
yhat = lda.predict(X_test)#
errl=sum(Y_test!=yhat)/len(Y_test) #
print("Taux d'erreur: ",round(errl,3))
plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(Y_test,yhat) #
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(Y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()

score = cross_val_score(lda, X_train, Y_train, cv=cv, scoring=make_scorer(f1_score, pos_label="BRCA", zero_division=0), error_score="raise")
print("Accuracy moyenne en CV sur le train :", np.mean(score))


### KPPV ----------------------------------------------------------------------
print("### KPPV ####")

# La standardisation et le filtrage sont réappris dans chaque pli de CV.
knn_pipeline = Pipeline([
    ("variance", VarianceThreshold(threshold=1e-16)),
    ("scaler", StandardScaler()),
    ("knn", neighbors.KNeighborsClassifier()),
])

# Même première référence que dans le notebook d’origine : k = 10.
knn_pipeline.set_params(knn__n_neighbors=10)
knn_pipeline.fit(X_train, Y_train)
y_pred = knn_pipeline.predict(X_test)
print("Matrice de confusion pour k = 10 :")
print(confusion_matrix(Y_test, y_pred))
print("Accuracy :", round(accuracy_score(Y_test, y_pred), 3))
print("F1 BRCA :", round(f1_score(Y_test, y_pred, pos_label="BRCA", zero_division=0), 3))

from sklearn.metrics import roc_curve, RocCurveDisplay
y_score = knn_pipeline.predict_proba(X_test)

colonne_brca = list(knn_pipeline.classes_).index("BRCA")
fpr, tpr, _ = roc_curve(Y_test == "BRCA", y_score[:, colonne_brca])
RocCurveDisplay(fpr=fpr, tpr=tpr).plot()
plt.title("Courbe ROC BRCA — k = 10")
plt.show()

# Sélection de k uniquement sur le train.
search_knn = GridSearchCV(
    knn_pipeline,
    {"knn__n_neighbors": [1, 3, 5, 10, 20]},
    scoring=make_scorer(f1_score, pos_label="BRCA", zero_division=0),
    cv=cv,
    n_jobs=N_JOBS,
    pre_dispatch=N_JOBS,
    error_score="raise",
)
search_knn.fit(X_train, Y_train)
print("Valeurs de k :", search_knn.cv_results_["param_knn__n_neighbors"])
print("Scores CV moyens :", search_knn.cv_results_["mean_test_score"])
print("Écarts-types CV :", search_knn.cv_results_["std_test_score"])
print("Meilleur k :", search_knn.best_params_)

knn = search_knn.best_estimator_
y_pred = knn.predict(X_test)
print("Matrice de confusion pour le k retenu :")
print(confusion_matrix(Y_test, y_pred))
print("Accuracy :", round(accuracy_score(Y_test, y_pred), 3))
print("F1 BRCA :", round(f1_score(Y_test, y_pred, pos_label="BRCA", zero_division=0), 3))
print("Précision BRCA :", round(precision_score(Y_test, y_pred, pos_label="BRCA", zero_division=0), 3))
print("Rappel BRCA :", round(recall_score(Y_test, y_pred, pos_label="BRCA", zero_division=0), 3))
ConfusionMatrixDisplay.from_predictions(
    Y_test, y_pred, cmap="YlOrBr", colorbar=False
)
plt.show()

### Arbre -------------------------------------------------------------------------

print("### ARBRE POUR DEUX CLASSES ###")
clf = DecisionTreeClassifier(max_depth=4, random_state=nb_binome)
clf = clf.fit(X_train, Y_train)

plt.figure(figsize=(30,30))
plot_tree(clf, filled=True, feature_names=feature_names, class_names=list(clf.classes_))
plt.show()

y_pred_clf = clf.predict(X_test)
# print("Confusion matrix (proportions) : ")
# print(confusion_matrix(Y_test,y_pred_clf,normalize='True'))
# print(' ')
print("Confusion matrix : ")
cm = confusion_matrix(Y_test,y_pred_clf)
print(np.round(cm,2))
print(' ')

plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(Y_test,y_pred_clf)
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(Y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()

print("Accuracy : ",np.round(accuracy_score(Y_test,y_pred_clf),3))
print("F1-score BRCA: ",np.round(f1_score(Y_test,y_pred_clf,average="binary", pos_label="BRCA", zero_division=0),3))
print("precision_score BRCA: ",np.round(precision_score(Y_test,y_pred_clf,average="binary", pos_label="BRCA", zero_division=0),3))
print("recall_score: ",np.round(recall_score(Y_test,y_pred_clf,average="binary", pos_label="BRCA", zero_division=0),3))

d = {'X0': X_train[:,3], 'X1': X_train[:,6876], 'Label':Y_train}
df = pd.DataFrame(data=d)

sns.scatterplot(x="X0",y="X1",hue="Label",data = df)

## gridseachcv
# Pas de nouveau découpage pour la recherche d’hyperparamètres.

B = 15
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "max_depth": [4, 8, None],
    "min_samples_leaf": [1, 4, 8],
    "max_features": [None, "sqrt"],
    "splitter": ["best"],
}
clf = DecisionTreeClassifier(random_state=nb_binome)
search = GridSearchCV(clf,
                      param_grid,
                      scoring=make_scorer(f1_score, pos_label="BRCA", zero_division=0),
                      cv=cv,
                      n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score="raise")
search.fit(X_train, Y_train)
print('Meilleur modèle')
print(search.best_estimator_)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label="BRCA", zero_division=0),
    "recall": make_scorer(recall_score, pos_label="BRCA", zero_division=0),
    "f1": make_scorer(f1_score, pos_label="BRCA", zero_division=0),
    "roc_auc": "roc_auc",
}

tree = search.best_estimator_

plt.figure(figsize=(30,30))
plot_tree(tree, filled=True, feature_names=feature_names, class_names=list(tree.classes_))
plt.show()
plt.rcdefaults()


y_pred_tree = tree.predict(X_test)
print("Confusion matrix  : ")
print(confusion_matrix(Y_test,y_pred_tree))
print(' ')
print("Confusion matrix (proportions) : ")
cm = confusion_matrix(Y_test,y_pred_tree,normalize='true')
print(np.round(cm,2))
print(' ')

y_proba = tree.predict_proba(X_test)

plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(Y_test,y_pred_tree)
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(Y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()

print("Accuracy : ",np.round(accuracy_score(Y_test,y_pred_tree),3))
print("F1-score BRCA: ",np.round(f1_score(Y_test,y_pred_tree,average="binary", pos_label="BRCA", zero_division=0),3))
print("precision_score BRCA: ",np.round(precision_score(Y_test,y_pred_tree,average="binary", pos_label="BRCA", zero_division=0),3))
print("recall_score: ",np.round(recall_score(Y_test,y_pred_tree,average="binary", pos_label="BRCA", zero_division=0),3))

#### FOret aleatoire ------------------------------------------------------------


param_grid = {
    "max_depth": [5, None],
    "min_samples_leaf": [1, 4],
    "max_features": ["sqrt", 0.05],
}

rf = RandomForestClassifier(random_state=nb_binome)
search = GridSearchCV(rf,
                       param_grid,
                       scoring=make_scorer(f1_score, pos_label="BRCA", zero_division=0),
                       cv=cv,
                       n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score="raise")
search.fit(X_train,Y_train)
print('Meilleur modèle')
print(search.best_estimator_)


rf_best = search.best_estimator_
cv_scores = cross_validate(rf_best, X_train, Y_train, cv=cv, scoring=scoring, error_score="raise")

print("Scores CV sur le train après sélection des paramètres")
print("Accuracy  : ", np.round(np.mean(cv_scores["test_accuracy"]),2), "(",np.round(np.std(cv_scores["test_accuracy"]),2),")")
print("Precision : ", np.round(np.mean(cv_scores["test_precision"]),2), "(",np.round(np.std(cv_scores["test_precision"]),2),")")
print("Recall    : ", np.round(np.mean(cv_scores["test_recall"]),2), "(",np.round(np.std(cv_scores["test_recall"]),2),")")
print("F1-score  : ", np.round(np.mean(cv_scores["test_f1"]),2), "(",np.round(np.std(cv_scores["test_f1"]),2),")")
print("AUC       : ", np.round(np.mean(cv_scores["test_roc_auc"]),2), "(",np.round(np.std(cv_scores["test_roc_auc"]),2),")")

rf_best = search.best_estimator_
importances = rf_best.feature_importances_
indices = np.argsort(importances)[-20:]

#Plot the feature importances of the forest
plt.figure(figsize=(9,8))
plt.title("20 gènes les plus importants dans cette forêt")
plt.barh(feature_names[indices], importances[indices], color="r", align="center")
plt.show()
plt.rcdefaults()
# # # If you want to define your own labels,
# # # change indices to a list of labels on the following line.
#plt.yticks(range(X.shape[1]), data.columns[indices])
#plt.ylim([-1, X.shape[1]])
#plt.savefig("RF_importances.png",format="png")
plt.show()

from sklearn.metrics import ConfusionMatrixDisplay
y_pred_rf = rf_best.predict(X_test)

print("Matrice de confusion normalisée de la forêt aléatoire :")
cm_rf = confusion_matrix(Y_test,y_pred_rf,normalize='true')

ConfusionMatrixDisplay(cm_rf, display_labels=np.unique(Y_train)).plot()
plt.title("Random Forest")

plt.rcParams.update({'figure.figsize': (3,3),'font.size': 10})
conf_mat =  confusion_matrix(Y_test,y_pred_rf)
ConfusionMatrixDisplay(conf_mat, display_labels=np.unique(Y_train)).plot(cmap='YlOrBr',colorbar=False)
plt.show()
plt.rcdefaults()


print("Accuracy : ",np.round(accuracy_score(Y_test,y_pred_rf),3))
print("F1-score BRCA: ",np.round(f1_score(Y_test,y_pred_rf,average="binary", pos_label="BRCA", zero_division=0),3))
print("precision_score BRCA: ",np.round(precision_score(Y_test,y_pred_rf,average="binary", pos_label="BRCA", zero_division=0),3))
print("recall_score: ",np.round(recall_score(Y_test,y_pred_rf,average="binary", pos_label="BRCA", zero_division=0),3))

### Adaboost -------------------------------------------------------------------
print("### AdaBoost ###")

# L’ACP est ajustée sur chaque pli d’entraînement, jamais sur le test.
boost_pipeline = Pipeline([
    ("variance", VarianceThreshold(threshold=1e-16)),
    ("pca", PCA(n_components=10, svd_solver="randomized", random_state=nb_binome)),
    ("boost", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, random_state=nb_binome),
        random_state=nb_binome,
    )),
])
param_grid = {
    "boost__n_estimators": [100, 200, 300],
    "boost__learning_rate": [0.01, 0.1, 1.0],
}
search = GridSearchCV(
    boost_pipeline, param_grid,
    scoring=make_scorer(f1_score, pos_label="BRCA", zero_division=0), cv=cv,
    n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score="raise",
)
search.fit(X_train, Y_train)
print("Meilleurs paramètres :", search.best_params_)
boost_best = search.best_estimator_
y_pred_boost = boost_best.predict(X_test)

print("Matrice de confusion :")
print(confusion_matrix(Y_test, y_pred_boost))
cm_boost = confusion_matrix(Y_test, y_pred_boost, normalize="true")
ConfusionMatrixDisplay.from_predictions(
    Y_test, y_pred_boost, cmap="YlOrBr", colorbar=False
)
plt.title("AdaBoost")
plt.show()
print("Accuracy :", round(accuracy_score(Y_test, y_pred_boost), 3))
print("F1 BRCA :", round(f1_score(Y_test, y_pred_boost, pos_label="BRCA", zero_division=0), 3))
print("Précision BRCA :", round(precision_score(Y_test, y_pred_boost, pos_label="BRCA", zero_division=0), 3))
print("Rappel BRCA :", round(recall_score(Y_test, y_pred_boost, pos_label="BRCA", zero_division=0), 3))
